# Project 2 – Portfolio Optimization
**Group #2**

| Sector | Tickers |
|---|---|
| Financial | GS, MS, SCHW |
| Healthcare | JNJ, ABBV, TMO |
| Energy | XOM, SLB, EOG |
| Consumer | COST, NKE, SBUX |
| Industrial | CAT, DE, UPS |
| Technology | AMD, ORCL, CRM, CMCSA, LIN |

**Period:** 2017-01-01 → 2023-12-31

**Requirements:** Data download · 1/N baseline · CVXPY optimisation · Efficient frontier (γ sensibilisation + scipy) · Special portfolios · Weight tables · Leverage / short-selling (extra +10)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import cvxpy as cp
from scipy.optimize import minimize
import yfinance as yf

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

# ── Configuration ─────────────────────────────────────────────────────────────
TICKERS = [
    "GS",    # Goldman Sachs
    "MS",    # Morgan Stanley
    "SCHW",  # Charles Schwab
    "JNJ",   # Johnson & Johnson
    "ABBV",  # AbbVie
    "TMO",   # Thermo Fisher Scientific
    "XOM",   # Exxon Mobil
    "SLB",   # Schlumberger
    "EOG",   # EOG Resources
    "COST",  # Costco
    "NKE",   # Nike
    "SBUX",  # Starbucks
    "CAT",   # Caterpillar
    "DE",    # Deere & Company
    "UPS",   # United Parcel Service
    "AMD",   # Advanced Micro Devices
    "ORCL",  # Oracle
    "CRM",   # Salesforce
    "CMCSA", # Comcast
    "LIN",   # Linde
]
START_DATE   = "2017-01-01"
END_DATE     = "2023-12-31"
RISK_FREE    = 0.03    # annual risk-free rate (≈ avg 3-month T-bill 2017-2023)
TRADING_DAYS = 252
n            = len(TICKERS)
print(f"{n} assets loaded.")

## 1. Download Asset Prices

In [ ]:
prices = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True)["Close"]
prices.dropna(how="any", inplace=True)  # keep only dates where all assets traded

print(f"Trading days loaded : {len(prices)}")
print(f"Date range          : {prices.index[0].date()} → {prices.index[-1].date()}")
print(f"Assets              : {len(prices.columns)}")
prices.tail()

In [ ]:
# Daily log-returns (preferred over simple returns for MVO: additive over time)
log_returns = np.log(prices / prices.shift(1)).dropna()

# Annualised expected returns (μ) and covariance matrix (Σ)
mu    = log_returns.mean().values * TRADING_DAYS   # shape (n,)
Sigma = log_returns.cov().values  * TRADING_DAYS   # shape (n, n)

# Quick sanity check
df_stats = pd.DataFrame({
    "Ann. Return (%)":    (mu * 100).round(2),
    "Ann. Volatility (%)": (np.sqrt(np.diag(Sigma)) * 100).round(2),
}, index=TICKERS)
df_stats

## 2. 1/N Equally-Weighted Portfolio (Baseline)

In [ ]:
w_eq      = np.ones(n) / n
ret_eq    = float(w_eq @ mu)
risk_eq   = float(np.sqrt(w_eq @ Sigma @ w_eq))
sharpe_eq = (ret_eq - RISK_FREE) / risk_eq

print(f"Weight per asset  : {w_eq[0]*100:.2f}%  (equal for all {n} assets)")
print(f"Annual Return     : {ret_eq*100:.2f}%")
print(f"Annual Volatility : {risk_eq*100:.2f}%")
print(f"Sharpe Ratio      : {sharpe_eq:.4f}")

**Commentary – 1/N portfolio**

The 1/N portfolio returned **13.55%** annually at **20.80%** volatility, giving a Sharpe ratio of **0.51**. It lands visibly to the right of the efficient frontier in the plot — meaning an optimised portfolio can achieve the same return at lower risk. The max-Sharpe portfolio (Sharpe = 0.98) nearly doubles it on a risk-adjusted basis, so optimisation does add clear value here.

That said, the 1/N portfolio is still a useful baseline: it requires no estimation of expected returns, avoids concentration risk, and would be easy to maintain with quarterly rebalancing. Its weakness in this dataset is that it equally weights ORCL (−4.1% annual return) alongside MS (+36.6%), diluting performance unnecessarily.

## 3. Optimization Problem Definition

We solve the classic **Markowitz Mean-Variance Optimization (MVO)** in two equivalent formulations:

### (a) Risk-aversion form  *(used with CVXPY for the frontier)*

$$\max_{w} \; \mu^\top w \;-\; \gamma \, w^\top \Sigma \, w$$

$$\text{s.t.} \quad \mathbf{1}^\top w = 1, \quad w_i \ge 0 \; \forall i$$

- $w \in \mathbb{R}^n$: portfolio weight vector
- $\mu \in \mathbb{R}^n$: annualised expected returns
- $\Sigma \in \mathbb{R}^{n \times n}$: annualised covariance matrix (PSD)
- $\gamma \ge 0$: **risk-aversion coefficient** (sensitivity parameter)
  - $\gamma \to 0$: the solver chases maximum return (concentrated portfolio)
  - $\gamma \to \infty$: the solver minimises variance (minimum-variance portfolio)
  - Sweeping $\gamma$ traces the entire efficient frontier.

### (b) Target-return form  *(used with scipy for cross-validation)*

$$\min_{w} \; w^\top \Sigma \, w$$

$$\text{s.t.} \quad \mu^\top w = r^*, \quad \mathbf{1}^\top w = 1, \quad w_i \ge 0 \; \forall i$$

Both formulations are **convex programs** — the feasible set is a convex polytope and the objective is quadratic — so CVXPY (SCS solver) and scipy (SLSQP) guarantee global optima.

## 4. Efficient Frontier

### 4a. CVXPY — Risk-Aversion Form (γ Sensibilisation)

In [ ]:
def cvxpy_frontier(mu, Sigma, n, gammas, lb=0.0):
    # lb: lower bound on weights (0=long-only, -L=short-selling allowed)
    # Returns: (risks, returns, weights_list)
    w         = cp.Variable(n)
    gamma_par = cp.Parameter(nonneg=True)
    objective   = cp.Maximize(mu @ w - gamma_par * cp.quad_form(w, Sigma))
    constraints = [cp.sum(w) == 1, w >= lb]
    prob        = cp.Problem(objective, constraints)

    risks, returns, weights = [], [], []
    for g in gammas:
        gamma_par.value = g
        prob.solve(solver=cp.SCS, warm_start=True, verbose=False)
        if prob.status in ("optimal", "optimal_inaccurate") and w.value is not None:
            wv = w.value
            risks.append(float(np.sqrt(wv @ Sigma @ wv)))
            returns.append(float(mu @ wv))
            weights.append(wv.copy())

    return np.array(risks), np.array(returns), weights

# Sweep gamma over 4 orders of magnitude → captures full frontier
gammas_fine = np.logspace(-2, 4, 300)
risks_cvx, rets_cvx, weights_cvx = cvxpy_frontier(mu, Sigma, n, gammas_fine, lb=0.0)

print(f"Portfolios solved : {len(risks_cvx)}")
print(f"Return range      : [{rets_cvx.min()*100:.2f}%, {rets_cvx.max()*100:.2f}%]")
print(f"Volatility range  : [{risks_cvx.min()*100:.2f}%, {risks_cvx.max()*100:.2f}%]")

**Commentary – γ sensibilisation**

| γ | Behaviour | What happens in this dataset |
|---|---|---|
| ~0.01 | Variance barely penalised | 100% MS (highest μ = 36.6%, but vol = 55.7%) |
| ~1–10 | Balanced | Gradually diversifies across ABBV, GS, NKE, XOM |
| ~1000+ | Variance dominates | Converges to min-variance: COST 38.5%, ABBV 19.4% |

Two things are visible in the left plot:
- The **CVXPY frontier** (blue) traces only the efficient upper branch — at each γ the solver maximises return for a given risk, so it never touches the lower (inefficient) half of the parabola.
- The **scipy frontier** (green dashed) sweeps all target returns from ORCL's −4.1% up to MS's 36.6%, so it traces both branches of the mean-variance parabola. The downward-bending lower portion below the min-variance point is clearly visible. These are inefficient portfolios (same risk, lower return) and should never be chosen in practice.

The two methods agree well on the efficient portion, which validates the implementation.

### 4b. scipy — Target-Return Sweep (cross-validation)

In [ ]:
def scipy_frontier(mu, Sigma, n, n_points=200):
    # Sweep target returns and minimise variance via scipy SLSQP.
    targets = np.linspace(mu.min(), mu.max(), n_points)
    risks, returns, weights = [], [], []
    w0 = np.ones(n) / n

    for r_target in targets:
        cons = [
            {"type": "eq", "fun": lambda w: np.sum(w) - 1},
            {"type": "eq", "fun": lambda w, r=r_target: w @ mu - r},
        ]
        res = minimize(
            lambda w: w @ Sigma @ w, w0,
            method="SLSQP", bounds=[(0.0, 1.0)] * n,
            constraints=cons, options={"ftol": 1e-12, "maxiter": 1000},
        )
        if res.success:
            wv = res.x
            risks.append(float(np.sqrt(wv @ Sigma @ wv)))
            returns.append(float(wv @ mu))
            weights.append(wv.copy())

    return np.array(risks), np.array(returns), weights

risks_sp, rets_sp, weights_sp = scipy_frontier(mu, Sigma, n)
print(f"scipy portfolios solved: {len(risks_sp)}")

## 5. Special Portfolios

In [ ]:
# ── Min-Variance ─────────────────────────────────────────────────────────────
w_cv = cp.Variable(n)
cp.Problem(cp.Minimize(cp.quad_form(w_cv, Sigma)),
           [cp.sum(w_cv) == 1, w_cv >= 0]).solve(solver=cp.SCS, verbose=False)
w_minvar    = w_cv.value
ret_minvar  = float(w_minvar @ mu)
risk_minvar = float(np.sqrt(w_minvar @ Sigma @ w_minvar))
sharpe_minvar = (ret_minvar - RISK_FREE) / risk_minvar

# ── Max Return (long-only → 100% in highest-μ asset) ─────────────────────────
idx_max   = np.argmax(mu)
w_maxret  = np.zeros(n); w_maxret[idx_max] = 1.0
ret_maxret  = float(w_maxret @ mu)
risk_maxret = float(np.sqrt(w_maxret @ Sigma @ w_maxret))
sharpe_maxret = (ret_maxret - RISK_FREE) / risk_maxret

# ── Min Return (long-only → 100% in lowest-μ asset) ──────────────────────────
idx_min   = np.argmin(mu)
w_minret  = np.zeros(n); w_minret[idx_min] = 1.0
ret_minret  = float(w_minret @ mu)
risk_minret = float(np.sqrt(w_minret @ Sigma @ w_minret))
sharpe_minret = (ret_minret - RISK_FREE) / risk_minret

# ── Max Sharpe Ratio (numerical, 100 random starts) ──────────────────────────
def neg_sharpe(w, mu, Sigma, rf):
    r = np.sqrt(w @ Sigma @ w)
    return -(w @ mu - rf) / r if r > 1e-10 else 1e10

np.random.seed(42)
best, w_maxsr = np.inf, None
for _ in range(100):
    w0  = np.random.dirichlet(np.ones(n))
    res = minimize(neg_sharpe, w0, args=(mu, Sigma, RISK_FREE),
                   method="SLSQP", bounds=[(0, 1)] * n,
                   constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1}],
                   options={"ftol": 1e-12, "maxiter": 2000})
    if res.success and res.fun < best:
        best, w_maxsr = res.fun, res.x.copy()

ret_maxsr   = float(w_maxsr @ mu)
risk_maxsr  = float(np.sqrt(w_maxsr @ Sigma @ w_maxsr))
sharpe_maxsr = (ret_maxsr - RISK_FREE) / risk_maxsr

# ── Summary table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    "Annual Return (%)":    [ret_eq, ret_minvar, ret_maxret, ret_minret, ret_maxsr],
    "Annual Volatility (%)": [risk_eq, risk_minvar, risk_maxret, risk_minret, risk_maxsr],
    "Sharpe Ratio":         [sharpe_eq, sharpe_minvar, sharpe_maxret, sharpe_minret, sharpe_maxsr],
}, index=["1/N Equal Weight", "Min Variance", "Max Return", "Min Return", "Max Sharpe Ratio"])

summary[["Annual Return (%)", "Annual Volatility (%)"]] *= 100
summary = summary.round(4)
summary

**Commentary – Special Portfolios**

- **Min Variance (Sharpe 0.57, vol 16.26%):** COST takes 38.54% — the lowest individual volatility in the set (19.25%). ABBV gets 19.41% because its return (22.48%) is high relative to its volatility (22.95%), making it efficient at reducing portfolio risk without sacrificing much return. The remaining 42% is spread across GS, LIN, CRM, JNJ, CMCSA, NKE, DE, and UPS — all relatively low-vol names. Notably, MS, SLB, ORCL, and AMD are completely excluded due to their high volatility.

- **Max Return (Sharpe 0.60, vol 55.72%):** 100% Morgan Stanley. MS had an outsized return of 36.63% over this period, likely driven by the post-COVID financial sector recovery and strong investment banking activity in 2020-2021. The 55.72% volatility reflects its sensitivity to market cycles. This portfolio is not investable in practice.

- **Min Return (Sharpe −0.16, vol 44.20%):** 100% Oracle (ORCL), which was the only asset with a **negative** annualised return (−4.10%) over the period. ORCL significantly underperformed between 2017 and 2022 before recovering late, but not enough to offset cumulative losses. Its 44.20% volatility with negative return gives the only negative Sharpe ratio in the set.

- **Max Sharpe (Sharpe 0.98, vol 19.45%):** ABBV dominates at 48.21% — it has an attractive combination of 22.48% return and only 22.95% volatility, making it the most efficient individual asset. GS (15.82%), NKE (14.10%), XOM (8.25%), MS (8.11%), and CRM (5.51%) complete the portfolio. The optimiser deliberately avoids high-vol/low-return assets like SLB, AMD, TMO, and ORCL entirely. At Sharpe 0.98, this portfolio nearly doubles the 1/N benchmark (0.51).

## 6. Weight Tables for Special Portfolios

In [ ]:
df_weights = pd.DataFrame({
    "1/N Equal Weight" : w_eq,
    "Min Variance"     : w_minvar,
    "Max Return"       : w_maxret,
    "Min Return"       : w_minret,
    "Max Sharpe Ratio" : w_maxsr,
}, index=TICKERS)

(df_weights * 100).round(2).style \
    .format("{:.2f}%") \
    .background_gradient(axis=0, cmap="YlGn") \
    .set_caption("Portfolio Weights (%)")

In [ ]:
# Significant holdings only (weight > 1%) — easier to interpret
print("Significant holdings (weight > 1%) by portfolio:\n")
for col in df_weights.columns:
    sig = (df_weights[col] * 100)
    sig = sig[sig > 1.0].sort_values(ascending=False)
    print(f"  {col}:")
    for ticker, wt in sig.items():
        print(f"    {ticker:6s} {wt:6.2f}%")
    print()

## 7. Leverage / Short Selling  *(Extra +10 pts)*

We parameterise leverage by a **lower bound $L \ge 0$** on individual weights:

$$w_i \ge -L \quad \forall i, \qquad \mathbf{1}^\top w = 1$$

- $L = 0$: long-only (no short selling)
- $L > 0$: each asset may be shorted up to $L \times$ total capital
- Gross leverage: $\sum_i |w_i| = 1 + 2\sum_i \max(0, -w_i) \le 1 + 2nL$

This is the formulation described in the textbook (p. 402). The net investment constraint ($\sum w_i = 1$) ensures we stay fully invested.

In [ ]:
leverage_levels = {
    "L=0  (no leverage)"     : 0.0,
    "L=0.3 (mild leverage)"  : 0.3,
    "L=0.6 (med leverage)"   : 0.6,
    "L=1.0 (high leverage)"  : 1.0,
}

frontier_by_lev = {}
for label, L in leverage_levels.items():
    r, ret, w = cvxpy_frontier(mu, Sigma, n, gammas_fine, lb=-L)
    frontier_by_lev[label] = (r, ret, w)
    print(f"{label}: {len(r)} pts | "
          f"ret ∈ [{ret.min()*100:.1f}%, {ret.max()*100:.1f}%] | "
          f"vol ∈ [{r.min()*100:.1f}%, {r.max()*100:.1f}%]")

**Commentary – Leverage**

The outputs show dramatic frontier expansion with leverage:

| Level | Max return | Max vol |
|---|---|---|
| L=0 (no leverage) | 36.6% | 55.7% |
| L=0.3 | 175.1% | 335.5% |
| L=0.6 | 313.6% | 618.2% |
| L=1.0 | 498.3% | 995.4% |

These extreme values come from the low-γ end of the sweep, where the optimizer goes massively long in MS (best return at 36.6%) and simultaneously short in ORCL (worst at −4.1%), amplifying the return spread. At L=1.0 with γ≈0, individual positions can reach hundreds of times the portfolio value.

**What the right-hand plot actually shows:** The no-leverage frontier (blue) is barely visible — it's compressed into the lower-left corner by the scale of the leveraged frontiers. The practical, investable region is the **left portion** of each curve (low volatility, moderate return), where all four frontiers cluster near 15–16% volatility and diverge only gradually. That is the only region worth examining for real allocation decisions.

**Key conclusions on leverage:**
1. The theoretical return improvement is real but comes with extreme concentration risk — the high-return portfolios are essentially leveraged single-stock bets on MS vs. ORCL.
2. Borrowing costs, margin requirements, and short-squeeze risk are not modelled. In practice they would significantly erode the theoretical gains.
3. The minimum-volatility region (left of the frontier) improves only modestly with leverage — from ~16.3% vol at L=0 to ~15.9% at L=0.3 — suggesting that for risk-averse investors the benefit of short selling is limited in this universe.

## Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle("Project 2 – Portfolio Optimization  |  20 Assets  |  2017-2023",
             fontsize=14, fontweight="bold")

# ── Left: efficient frontiers + special portfolios ────────────────────────────
ax = axes[0]
ax.plot(risks_cvx * 100, rets_cvx * 100, "b-",  lw=2.5,
        label="CVXPY frontier (γ sweep)", zorder=3)
ax.plot(risks_sp  * 100, rets_sp  * 100, "g--", lw=1.8,
        label="scipy frontier (target-return)", zorder=3)

specials = [
    (risk_minvar,  ret_minvar,  "Min Variance",    "s", "purple", 130),
    (risk_maxret,  ret_maxret,  "Max Return",       "^", "red",    130),
    (risk_minret,  ret_minret,  "Min Return",       "v", "orange", 130),
    (risk_maxsr,   ret_maxsr,   "Max Sharpe",       "*", "gold",   220),
    (risk_eq,      ret_eq,      "1/N Equal Weight", "o", "cyan",   130),
]
for risk, ret, label, mk, col, sz in specials:
    ax.scatter(risk*100, ret*100, marker=mk, s=sz, color=col,
               edgecolors="black", lw=0.8, label=label, zorder=5)
    ax.annotate(label, (risk*100, ret*100),
                xytext=(8, 4), textcoords="offset points", fontsize=7.5)

for i, ticker in enumerate(TICKERS):
    ri = np.sqrt(Sigma[i, i]) * 100
    ax.scatter(ri, mu[i]*100, marker="x", s=40, color="gray", alpha=0.55, zorder=2)
    ax.annotate(ticker, (ri, mu[i]*100),
                xytext=(3, 2), textcoords="offset points", fontsize=6.5, color="gray")

ax.set_xlabel("Annual Volatility (%)", fontsize=11)
ax.set_ylabel("Annual Return (%)",     fontsize=11)
ax.set_title("Efficient Frontier + Special Portfolios", fontsize=12)
ax.legend(fontsize=8, loc="lower right")
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mtick.FormatStrFormatter("%.0f%%"))
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.0f%%"))

# ── Right: leverage frontiers ─────────────────────────────────────────────────
ax2 = axes[1]
colors_lev = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]

for (label, (r, ret, _)), col in zip(frontier_by_lev.items(), colors_lev):
    ax2.plot(r*100, ret*100, lw=2.2, label=label, color=col)

ax2.axhline(RISK_FREE * 100, color="black", ls=":", lw=1.2,
            label=f"Risk-free rate ({RISK_FREE*100:.0f}%)")

ax2.set_xlabel("Annual Volatility (%)", fontsize=11)
ax2.set_ylabel("Annual Return (%)",     fontsize=11)
ax2.set_title("Efficient Frontiers by Leverage Level\n"
              r"(Short-selling: $w_i \geq -L$)", fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mtick.FormatStrFormatter("%.0f%%"))
ax2.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.0f%%"))

plt.tight_layout()
plt.savefig("efficient_frontier.png", dpi=180, bbox_inches="tight")
plt.show()
print("Plot saved → efficient_frontier.png")

## Conclusions

### 1. Asset Universe (2017-2023)
The 20 assets show wide dispersion in outcomes. MS (+36.6%) and ABBV (+22.5%) were standout performers; ORCL was the only asset with a negative annualised return (−4.1%), underperforming despite being a technology name. High volatility was not rewarded uniformly: SLB (vol 43.8%, return 5.6%) and AMD (vol 35.8%, return 9.0%) both delivered poor risk-adjusted returns, while ABBV achieved the best individual Sharpe in the set.

### 2. 1/N Portfolio
The 1/N portfolio returned 13.55% at 20.80% volatility (Sharpe 0.51). It sits to the right of the efficient frontier, meaning it is not efficient — an optimised portfolio can achieve the same return with less risk. Its main weakness here is forced equal exposure to poor assets like ORCL and SLB alongside strong performers. However, it remains a reasonable baseline for investors who distrust return estimates.

### 3. Efficient Frontier
The two methods (CVXPY γ-sweep and scipy target-return) agree well on the efficient portion of the frontier, validating the implementation. The scipy method additionally traces the lower (inefficient) branch of the parabola, visible in the plot. The frontier spans from the min-variance portfolio (vol 16.3%, return 12.3%) up to MS (vol 55.7%, return 36.6%).

### 4. Special Portfolios
The optimisation results are clear and consistent:
- **Min Variance** is heavily concentrated in COST (38.5%) and ABBV (19.4%) — the two assets with the best volatility characteristics.
- **Max Sharpe** (Sharpe 0.98) selects ABBV (48.2%), GS, NKE, XOM, MS, and CRM — assets with high return relative to their individual and pairwise risk. It nearly doubles the Sharpe of the 1/N portfolio.
- The gap between the 1/N Sharpe (0.51) and the Max Sharpe (0.98) shows that in this universe, over this period, there was genuine alpha to be captured by optimisation.

### 5. Leverage
Short selling dramatically expands the theoretical frontier (up to 498% return and 995% vol at L=1.0), but these extremes are driven by leveraged bets on the MS-vs-ORCL return spread and are not practically investable. The useful part of the analysis is the low-volatility region (15–20% vol), where mild leverage (L=0.3) provides a small but real improvement. High leverage adds little in the risk-managed region and introduces serious estimation and operational risks.